# Transform Results Data

1. Read bronze `results` table
2. Keep only the columns required for Analytics (drop url column)
3. Standardise column names using snake_case
4. Concatenete `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform values to Title Case
5. Filter out rows where `driver_id` is null
6. Remove duplicated values
8. Write the transformated data to a silver table 

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.results'
silver_table = f'{catalog_name}.{silver_schema}.results'

## Step 1 - Read bronze `results` table


In [0]:
results_df = spark.read.table(bronze_table)

In [0]:
display(results_df)

## Step 2 - Keep only the columns required for Analytics (drop url column)

In [0]:
results_dropped_df = results_df.drop('url')

##Step 3 - Standardise column names using snake_case


In [0]:
results_df_renamed = results_dropped_df.withColumnsRenamed(
    {
        'driverId': 'driver_id',
        'constructorId': 'constructor_id',
        'positionText': 'finish_position_text',
        'raceName': 'race_name',
        'date': 'race_date',
        'grid': 'grid_position',
        'laps': 'completed_laps',
        'number': 'car_number',
        'position': 'finish_position'
    }
)

##Step 4 - Transform values to Title Case

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df_titled_case = results_df_renamed.withColumn('race_name', F.initcap('race_name'))

## Step 5 - Filter out rows where `driver_id` is null

In [0]:
results_df_not_null = results_df_titled_case.filter(
    (
        F.col('season').isNotNull() |
        F.col('constructor_id').isNotNull() |
        F.col('round').isNotNull() |
        F.col('season').isNotNull()
    )
)
results_df_not_null.count()

## Step 6 - Remove duplicated values

In [0]:
results_df_final = results_df_not_null.dropDuplicates(['driver_id', 'constructor_id', 'round', 'season'])
results_df_final.count()

## Step 7 - Write the transformted data to a silver table 

In [0]:
(
    results_df_final.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
SELECT * FROM formula1.silver.results